# Charge-exchange losses

A fast ion that meets a hydrogen atom of the background can capture its electron and leave the plasma as a fast neutral. The atoms come from recycling at the wall; their density $n_0$ is highest at the edge, typically $10^{15}$ to $10^{17}\ \mathrm{m^{-3}}$, and falls off inside the plasma. At low plasma density, where the fast ions take long to slow down, this is a loss channel for the beam power, and the fast neutrals that leave are what a neutral particle analyser (NPA) measures.

`slow_down_with_charge_exchange` adds the loss rate $\nu_\mathrm{cx} = n_0\,\sigma_\mathrm{cx}(E/A_b)\,v$ to the classical slowing down of `slow_down`. The cross section of H$^+$ + H(1s) $\to$ H + H$^+$ is the fit of Janev and Smith (IAEA, 1993) from plasmacoll, a function of the energy per nucleon (the relative speed), so a deuteron of 60 keV sees the cross section of a 30 keV proton. It falls steeply above about 20 keV/u:

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr
from plasmacoll.charge_exchange import proton_hydrogen_charge_exchange

import freeofcharge as foc
from freeofcharge.charge_exchange import (
    exponential_neutral_density,
    slow_down_with_charge_exchange,
)

plasma_plots.set_backend("plotly")  # interactive figures, embedded in the docs
plasma_plots.set_math("latex")  # formulas in the figures typeset as LaTeX

energy_per_nucleon = np.geomspace(100.0, 1e6, 120)  # eV/u
sigma = xr.DataArray(
    proton_hydrogen_charge_exchange(energy_per_nucleon),
    coords={
        "energy": (
            "energy",
            energy_per_nucleon / 1e3,
            {"long_name": "energy per nucleon", "units": "keV/u"},
        )
    },
    dims="energy",
    name="sigma",
    attrs={"long_name": r"$\sigma_\mathrm{cx}$", "units": "m^2"},
)
for e in (10e3, 30e3, 50e3):
    print(f"{e / 1e3:.0f} keV/u: {float(proton_hydrogen_charge_exchange(e)):.3g} m^2")
sigma.plasma.plot.lineout(
    x="energy", logx=True, logy=True, title="H+ + H(1s) charge exchange"
)

## The model

Each fast ion slows down on its birth surface, $\mathrm{d}v/\mathrm{d}t = -a_e v - a_i/v^2$, as in `slow_down`, and survives charge exchange down to the speed $v$ with the probability

$$
S(v) = \exp\left(-\int \nu_\mathrm{cx}\,\mathrm{d}t\right).
$$

All moments are weighted with $S$: the heating of electrons and ions, the fast-ion density, pressure and current. The power carried away by the fast neutrals is $P_\mathrm{cx} = \int S\,\nu_\mathrm{cx}\,\tfrac12 m_b v^2\,\mathrm{d}t$ per birth, and $P_e + P_i + P_\mathrm{th} + P_\mathrm{cx}$ equals the deposited power. Where $S$ falls steeply, the speed grid is refined, and trajectories are cut where $S < e^{-40}$.

The neutral density can be a number, one value per shell or a function of $\rho$; `exponential_neutral_density(n_edge, decay_length)` gives $n_0 = n_{0,\mathrm{edge}}\,e^{-(1-\rho)/\lambda}$.

The example is a medium-size tokamak ($R_0 = 1.65$ m, $a = 0.5$ m) at low density, $n_{e0} = 2\times10^{19}\ \mathrm{m^{-3}}$ and $T_{e0} = T_{i0} = 3$ keV, heated by a 60 keV, 2.5 MW deuterium beam. The neutrals have $n_{0,\mathrm{edge}} = 10^{16}\ \mathrm{m^{-3}}$ and decay over $\lambda = 0.1$ in $\rho$. The ions join the thermal plasma at $1.5\,T_i$ (`thermal_factor=1.5`): the charge exchange of a thermal ion is part of the thermal ions' energy balance, not a loss of beam power. A quarter of the deposited power leaves as fast neutrals, and the heating of electrons and ions drops accordingly:

In [ ]:
equilibrium = foc.Equilibrium.solovev(1.65, 0.5, 1.6, 2.5, 1.2, wall_psi_n=1.3)
plasma = foc.Plasma.parabolic(2e19, 3e3, ions={"D": (2, 1, 1.0)})
rho_edges = np.linspace(0.0, 1.0, 21)
surfaces = equilibrium.flux_surfaces(rho_edges)


def beam_deposition(energy, markers=1000):
    origin, direction = foc.tangential_injection(1.3, 3.5)
    beam = foc.NeutralBeam.gaussian(
        origin,
        direction,
        energy,
        2.5e6,
        width=0.1,
        divergence=0.01,
        markers=markers,
        seed=1,
    )
    return foc.deposit(
        beam,
        equilibrium,
        plasma,
        step=0.01,
        births_per_marker=2,
        rho_edges=rho_edges,
        seed=2,
    )


deposition = beam_deposition(60e3)
neutrals = exponential_neutral_density(1e16, 0.1)
thermal = {
    "thermal_factor": 1.5
}  # thermal below 1.5 T_i: no fast neutral from there on
without = foc.slow_down(deposition, plasma, surfaces, **thermal)
result = slow_down_with_charge_exchange(
    deposition, plasma, surfaces, neutrals, **thermal
)
steady = result.slowing_down
assert abs(result.power_balance) < 1e-9

print(
    f"deposited {steady.birth_power / 1e6:.2f} MW, thermalised {steady.thermalised_power / 1e6:.2f} MW"
)
print(
    f"lost to charge exchange {result.cx_power / 1e6:.2f} MW ({100 * result.lost_fraction:.0f} %)"
)
print(
    f"heating: electrons {steady.electron_heating.sum() / 1e6:.2f} MW, ions {steady.ion_heating.sum() / 1e6:.2f} MW"
    f" (without charge exchange {without.electron_heating.sum() / 1e6:.2f} and {without.ion_heating.sum() / 1e6:.2f} MW)"
)
print(
    f"fast-ion energy {steady.stored_energy / without.stored_energy:.2f} of the value without losses,"
)
print(f"driven current {steady.driven_current / without.driven_current:.2f} of it")

The charge-exchange probability $P_\mathrm{cx}$ of a fast ion rises from about 1 % for births in the core to nearly one outside $\rho \approx 0.7$: at this low density the ions slow down for a few hundred milliseconds, long enough to meet even the few neutrals at mid radius. The lost power, by birth shell, is largest around $\rho \approx 0.5$-0.7, where many ions are born and the neutrals already matter; the outermost shells have few births. The fast-ion pressure is reduced outside $\rho \approx 0.3$ and kept in the core:

In [ ]:
profiles = result.to_dataset(surfaces)
reference = without.to_dataset(surfaces)
profiles.cx_probability.plasma.plot.lineout(
    x="rho", title="charge-exchange probability of a fast ion"
).show()
profiles.cx_loss_density.plasma.plot.lineout(
    x="rho", title="charge-exchange loss by birth shell"
).show()
pressure = xr.concat(
    [reference.fast_ion_pressure, profiles.fast_ion_pressure],
    dim=xr.DataArray(["without neutrals", "with neutrals"], dims="case", name="case"),
)
pressure.plasma.plot.profiles(x="rho", over="case", title="fast-ion pressure")

## Scan of the edge neutral density

The lost share of the deposited power grows with the neutral density, from 7 % at $n_{0,\mathrm{edge}} = 10^{15}\ \mathrm{m^{-3}}$ to 25 % at $10^{16}$ and 56 % at $10^{17}\ \mathrm{m^{-3}}$; the fast-ion energy and the driven current fall with it, to about 55 % of their values without neutrals at $10^{17}\ \mathrm{m^{-3}}$:

In [ ]:
edge_density = np.geomspace(1e15, 1e17, 9)
scan = {"lost": [], "energy": [], "current": []}
for n_edge in edge_density:
    r = slow_down_with_charge_exchange(
        deposition,
        plasma,
        surfaces,
        exponential_neutral_density(n_edge, 0.1),
        **thermal,
    )
    scan["lost"].append(r.lost_fraction)
    scan["energy"].append(r.slowing_down.stored_energy / without.stored_energy)
    scan["current"].append(r.slowing_down.driven_current / without.driven_current)
density_scan = xr.Dataset(
    {
        "lost share of the beam power": ("n_edge", scan["lost"]),
        "fast-ion energy / without losses": ("n_edge", scan["energy"]),
        "driven current / without losses": ("n_edge", scan["current"]),
    },
    coords={
        "n_edge": (
            "n_edge",
            edge_density,
            {"long_name": r"$n_{0,\mathrm{edge}}$", "units": "1/m^3"},
        )
    },
).to_array("quantity", name="share")
for n_edge, lost in zip(edge_density[::4], scan["lost"][::4], strict=True):
    print(f"n_0,edge = {n_edge:.0e} m^-3: {100 * lost:.1f} % lost")
assert np.all(np.diff(scan["lost"]) > 0)
density_scan.plasma.plot.profiles(x="n_edge", over="quantity", logx=True, title="")

## Scan of the beam energy

A faster beam sees a smaller cross section and deposits deeper, but it also slows down for longer and passes through the energies of large cross sections on its way down. At $n_{0,\mathrm{edge}} = 10^{16}\ \mathrm{m^{-3}}$ the share of the deposited power lost to charge exchange falls only from 26 % at 30 keV to 22 % at 100 keV, while the shine-through (a share of the injected power) grows from 1 % to 12 %:

In [ ]:
beam_energy = np.array([30e3, 45e3, 60e3, 80e3, 100e3])
lost, shine = [], []
for energy in beam_energy:
    d = beam_deposition(energy, markers=300)
    r = slow_down_with_charge_exchange(d, plasma, surfaces, neutrals, **thermal)
    lost.append(r.lost_fraction)
    shine.append(d.shine_through_fraction)
    print(
        f"{energy / 1e3:.0f} keV: {100 * r.lost_fraction:.1f} % lost to charge exchange, {100 * d.shine_through_fraction:.1f} % shine-through"
    )
energy_scan = xr.Dataset(
    {
        "lost to charge exchange": ("energy", lost),
        "shine-through": ("energy", shine),
    },
    coords={
        "energy": (
            "energy",
            beam_energy / 1e3,
            {"long_name": "beam energy", "units": "keV"},
        )
    },
).to_array("loss", name="share of the injected or deposited power")
energy_scan.plasma.plot.profiles(x="energy", over="loss", title="")

## Fast-neutral spectrum

The fast neutrals leave with the energy the ion had at the capture. Their spectrum, summed over the plasma, is the emission an NPA samples (along its sightline, and without the re-ionisation of the neutrals on their way out, which this model leaves out). It is largest just below the birth energy and falls towards low energies, as the surviving ions thin out while they slow down; below $1.5\,T_i$ the ions are thermal and give none:

In [ ]:
edges = np.linspace(0.0, 62e3, 63)  # 1 keV bins
result = slow_down_with_charge_exchange(
    deposition, plasma, surfaces, neutrals, spectrum_edges=edges, **thermal
)
spectrum = result.to_dataset(surfaces).fast_neutral_spectrum * 1e3
spectrum = spectrum.assign_coords(energy=spectrum.energy / 1e3)
spectrum.energy.attrs.update(long_name="energy", units="keV")
spectrum.attrs.update(long_name="fast-neutral flux", units="1/(s keV)")
total = float(spectrum.sum())
assert abs(total / result.cx_rate.sum() - 1) < 1e-9
print(
    f"{total:.3g} fast neutrals per second; the largest flux at {float(spectrum.idxmax()):.1f} keV"
)
for e in (10, 30, 50):
    print(f"at {e} keV: {float(spectrum.sel(energy=e + 0.5)):.3g} per s and keV")
spectrum.plasma.plot.lineout(x="energy", title="fast neutrals leaving the plasma")

What the model leaves out: the orbit width (the ion sees the neutrals of its birth surface only; ions on wide orbits reach the edge from further in), the thermal speed of the neutrals, the re-ionisation of the fast neutrals (every one is counted as lost, an upper bound), charge exchange on the beam's own halo neutrals, and the excited states of the atoms. The neutral density itself is an input, from a neutral transport code or a measurement.